In [3]:
import os
from openai import OpenAI

client = OpenAI(
    # 各地域的API Key不同。获取API Key：https://help.aliyun.com/zh/model-studio/get-api-key
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # 以下是北京地域base-url，如果使用新加坡地域的模型，需要将base_url替换为：https://dashscope-intl.aliyuncs.com/compatible-mode/v1
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

resp = client.embeddings.create(
    model="text-embedding-v4",
    input=["喜欢，以后还来这里买"],
    # 将向量维度设置为 256
    dimensions=256
)
print(f"向量维度: {resp.data[0]}")

向量维度: Embedding(embedding=[0.010767209343612194, -0.19074776768684387, 0.04364610090851784, -0.08774397522211075, 0.04756145179271698, -0.012147621251642704, 0.08227252960205078, 0.0015670808497816324, -0.03190005570650101, 0.27166497707366943, 0.08493295311927795, -0.024847406893968582, 0.04695909097790718, 0.0058416505344212055, 0.016226109117269516, -0.09135814011096954, -0.008966400288045406, -0.0035639714915305376, -0.008552276529371738, -0.04419826716184616, 0.01273115910589695, 0.08719181269407272, 0.005509096663445234, 0.022726593539118767, -0.08237291872501373, -0.03187495842576027, -0.04648222029209137, -0.030143167823553085, 0.010472303256392479, 0.06410129368305206, -0.01657748781144619, -0.005474586505442858, -0.11013174057006836, 0.018698301166296005, 0.050472863018512726, 0.01416804175823927, -0.0025553300511091948, 0.08232272416353226, -0.014795501716434956, -0.006607151590287685, 0.0696229338645935, -0.050723847001791, 0.028561968356370926, 0.06736408174037933, -0.0349

In [ ]:
import dashscope
import numpy as np
from dashscope import TextEmbedding

def cosine_similarity(a, b):
    """计算余弦相似度"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_search(query, documents, top_k=5):
    """语义搜索"""
    # 生成查询向量
    query_resp = TextEmbedding.call(
        model="text-embedding-v4",
        input=query,
        dimension=1024
    )
    query_embedding = query_resp.output['embeddings'][0]['embedding']

    # 生成文档向量
    doc_resp = TextEmbedding.call(
        model="text-embedding-v4",
        input=documents,
        dimension=1024
    )

    # 计算相似度
    similarities = []
    for i, doc_emb in enumerate(doc_resp.output['embeddings']):
        similarity = cosine_similarity(query_embedding, doc_emb['embedding'])
        similarities.append((i, similarity))

    # 排序并返回top_k结果
    similarities.sort(key=lambda x: x[1], reverse=True)
    return [(documents[i], sim) for i, sim in similarities[:top_k]]

# 使用示例
documents = [
    "人工智能是计算机科学的一个分支",
    "机器学习是实现人工智能的重要方法",
    "深度学习是机器学习的一个子领域"
]
query = "什么是AI？"
results = semantic_search(query, documents, top_k=2)
for doc, sim in results:
    print(f"相似度: {sim:.3f}, 文档: {doc}")
